In [32]:
# Create Chunks and Filemap

import os
import hashlib
import json

# 64 KB
CHUNK_SIZE = 1024 * 64

# Generate hash for chunk.
def get_chunk_hash(chunk):
    hasher = hashlib.sha256()
    hasher.update(chunk)
    return hasher.hexdigest()

# Deduplicate files by storing only unique chunks.
def deduplicate_folder(input_folder, chunks_folder):
    if not os.path.exists(chunks_folder):
        os.makedirs(chunks_folder)
    
    chunk_map = {}
    file_map = {}

    for root, _, files in os.walk(input_folder):
        for filename in files:
            file_path = os.path.join(root, filename)
            chunk_list = []

            with open(file_path, 'rb') as f:
                while True:
                    chunk = f.read(CHUNK_SIZE)
                    if not chunk:
                        break

                    chunk_hash = get_chunk_hash(chunk)
                    
                    if chunk_hash not in chunk_map:
                        chunk_filename = os.path.join(chunks_folder, f'{chunk_hash}')
                        if not os.path.exists(chunk_filename):
                            with open(chunk_filename, 'wb') as chunk_file:
                                chunk_file.write(chunk)
                        chunk_map[chunk_hash] = chunk_filename

                    chunk_list.append(chunk_hash)

            relative_path = os.path.relpath(file_path, input_folder)
            file_map[relative_path] = chunk_list

    return file_map

def cleanup_chunks(file_map, chunks_folder):
    # Get all chunks used by existing files
    used_chunks = set()
    for chunks in file_map.values():
        used_chunks.update(chunks)

    # Check all existing chunks folder
    for chunk_file in os.listdir(chunks_folder):
        chunk_path = os.path.join(chunks_folder, chunk_file)
        if os.path.isfile(chunk_path):
            # If the chunk is not used by any file, delete it
            if chunk_file not in used_chunks:
                os.remove(chunk_path)
                

# Folders Path
input_folder = r'S:\Projects\Final year\deduplication\stuff'
chunks_folder = r'S:\Projects\Final year\deduplication\chunks'
file_map_path = r'S:\Projects\Final year\deduplication\filemap\file_map.json'

# Deduplicate the Folder
file_map = deduplicate_folder(input_folder, chunks_folder)

# Save the Filemap
os.makedirs(os.path.dirname(file_map_path), exist_ok=True)
with open(file_map_path, 'w') as f:
    json.dump(file_map, f, indent=4)

# Load the existing file map
if os.path.exists(file_map_path):
    with open(file_map_path, 'r') as f:
        existing_file_map = json.load(f)
else:
    existing_file_map = {}

# Remove entries for deleted files from the existing file map
updated_file_map = {k: v for k, v in existing_file_map.items() if os.path.exists(os.path.join(input_folder, k))}

# Cleanup chunks that are no longer used
cleanup_chunks(updated_file_map, chunks_folder)

# Save the updated file map
with open(file_map_path, 'w') as f:
    json.dump(updated_file_map, f, indent=4)

print("Deduplication Completed and Filemap saved.")

Deduplication Completed and Filemap saved.


In [27]:
# Restoration with Chunks and Filemap

import os
import json

# Paths
chunks_folder = r'S:\Projects\Final year\deduplication\chunks'
file_map_path = r'S:\Projects\Final year\deduplication\filemap\file_map.json'
restore_folder = r'S:\Projects\Final year\deduplication\restore'

def restore_files(file_map, chunks_folder, restore_folder):
    """Restore files from deduplicated chunks."""
    if not os.path.exists(restore_folder):
        os.makedirs(restore_folder)

    for relative_path, chunk_list in file_map.items():
        restore_path = os.path.join(restore_folder, relative_path)
        restore_dir = os.path.dirname(restore_path)
        if not os.path.exists(restore_dir):
            os.makedirs(restore_dir)

        with open(restore_path, 'wb') as f:
            for chunk_hash in chunk_list:
                chunk_filename = os.path.join(chunks_folder, chunk_hash)
                with open(chunk_filename, 'rb') as chunk_file:
                    f.write(chunk_file.read())

# Load the file map
with open(file_map_path, 'r') as f:
    file_map = json.load(f)

# Restore the files
restore_files(file_map, chunks_folder, restore_folder)

print("Restoration Completed.")

Restoration completed.


In [3]:
# Integrity check of stored chunks (Complete check)


import os
import hashlib
from tqdm import tqdm

# SHA-256 hash chunk
def compute_sha256(file_path):
    hasher = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 64)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()

# verify chunks 
def verify_chunks(chunks_folder):
    tampered_chunks = []
    chunk_files = [f for f in os.listdir(chunks_folder) if os.path.isfile(os.path.join(chunks_folder, f))]
    
    for chunk_name in tqdm(chunk_files, desc="Verifying Chunks"):
        chunk_path = os.path.join(chunks_folder, chunk_name)
        computed_hash = compute_sha256(chunk_path)
        if computed_hash != chunk_name:
            tampered_chunks.append(chunk_name)
    return tampered_chunks

# Path (chunks)
chunks_folder = r'S:\Projects\Final year\deduplication\chunks'
print("Complete Check")

tampered_chunks = verify_chunks(chunks_folder)
if tampered_chunks:
    print("The following chunks have been tampered with:")
    for chunk in tampered_chunks:
        print(chunk)
else:
    print("All chunks are intact.")

Complete Check


Verifying Chunks: 100%|████████████████████████████████████████████████████████████| 225/225 [00:00<00:00, 3291.78it/s]

All chunks are intact.


In [7]:
# Integrity check of stored chunks (Quick check 10%)

import os
import hashlib
import random
from tqdm import tqdm

# SHA-256 hash chunk
def compute_sha256(file_path):
    hasher = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 64)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()

# verify random chunks 
def verify_random_chunks(chunks_folder, sample_percentage=10):
    tampered_chunks = []
    chunk_files = [f for f in os.listdir(chunks_folder) if os.path.isfile(os.path.join(chunks_folder, f))]
    sample_size = max(1, len(chunk_files) * sample_percentage // 100)
    random_sample = random.sample(chunk_files, sample_size)
    
    for chunk_name in tqdm(random_sample, desc="Verifying Random Sample of Chunks"):
        chunk_path = os.path.join(chunks_folder, chunk_name)
        computed_hash = compute_sha256(chunk_path)
        if computed_hash != chunk_name:
            tampered_chunks.append(chunk_name)
    return tampered_chunks

# Path (chunks)
chunks_folder = r'S:\Projects\Final year\deduplication\chunks'
print("Quick Check")

tampered_chunks = verify_random_chunks(chunks_folder, sample_percentage=10)
if tampered_chunks:
    print("The following chunks have been tampered with:")
    for chunk in tampered_chunks:
        print(chunk)
else:
    print("Chunks are intact.")


Quick Check


Verifying Random Sample of Chunks: 100%|█████████████████████████████████████████████| 22/22 [00:00<00:00, 2823.93it/s]

Chunks are intact.


In [ ]:
#fastcdc implementation of the 1st code

import os
import hashlib
import json
import fastcdc

# Generate hash for chunk.
def get_chunk_hash(chunk):
    hasher = hashlib.sha256()
    hasher.update(chunk)
    return hasher.hexdigest()

# Deduplicate files by storing only unique chunks.
def deduplicate_folder(input_folder, chunks_folder):
    if not os.path.exists(chunks_folder):
        os.makedirs(chunks_folder)
    
    chunk_map = {}
    file_map = {}

    for root, _, files in os.walk(input_folder):
        for filename in files:
            file_path = os.path.join(root, filename)
            chunk_list = []

            with open(file_path, 'rb') as f:
                data = f.read()
                chunks = fastcdc.split(data, min_size=8192, avg_size=16384, max_size=65536)

                for chunk in chunks:
                    chunk_data = data[chunk.offset:chunk.offset + chunk.length]
                    chunk_hash = get_chunk_hash(chunk_data)
                    
                    if chunk_hash not in chunk_map:
                        chunk_filename = os.path.join(chunks_folder, f'{chunk_hash}')
                        if not os.path.exists(chunk_filename):
                            with open(chunk_filename, 'wb') as chunk_file:
                                chunk_file.write(chunk_data)
                        chunk_map[chunk_hash] = chunk_filename

                    chunk_list.append(chunk_hash)

            relative_path = os.path.relpath(file_path, input_folder)
            file_map[relative_path] = chunk_list

    return file_map

def cleanup_chunks(file_map, chunks_folder):
    # Get all chunks used by existing files
    used_chunks = set()
    for chunks in file_map.values():
        used_chunks.update(chunks)

    # Check all existing chunks folder
    for chunk_file in os.listdir(chunks_folder):
        chunk_path = os.path.join(chunks_folder, chunk_file)
        if os.path.isfile(chunk_path):
            # If the chunk is not used by any file, delete it
            if chunk_file not in used_chunks:
                os.remove(chunk_path)

# Folders Path
input_folder = r'S:\Projects\Final year\deduplication\stuff'
chunks_folder = r'S:\Projects\Final year\deduplication\chunks'
file_map_path = r'S:\Projects\Final year\deduplication\filemap\file_map.json'

# Deduplicate the Folder
file_map = deduplicate_folder(input_folder, chunks_folder)

# Save the Filemap
os.makedirs(os.path.dirname(file_map_path), exist_ok=True)
with open(file_map_path, 'w') as f:
    json.dump(file_map, f, indent=4)

# Load the existing file map
if os.path.exists(file_map_path):
    with open(file_map_path, 'r') as f:
        existing_file_map = json.load(f)
else:
    existing_file_map = {}

# Remove entries for deleted files from the existing file map
updated_file_map = {k: v for k, v in existing_file_map.items() if os.path.exists(os.path.join(input_folder, k))}

# Cleanup chunks that are no longer used
cleanup_chunks(updated_file_map, chunks_folder)

# Save the updated file map
with open(file_map_path, 'w') as f:
    json.dump(updated_file_map, f, indent=4)

print("Deduplication Completed and Filemap saved.")
